In [1]:
import asyncio

from agents import Agent, Runner, trace, function_tool
from rich.console import Console
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
system_prompt = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in return.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""

user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

In [3]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)


def get_report():
    result = ""
    for index, todo in enumerate(todos):
        if completions[index]:
            result += f"Todo #{index+1}: [green][strike]{todo}[/strike][green]\n"
        else:
            result += f"Todo #{index+1}: {todo}\n"
    show(result)
    return result

@function_tool
async def create_todos(descriptions: list[str]):
    """Add new todos from a list descriptions and return the full list.
    Args:
        descriptions: List of descriptions, each description is a string.
    """
    print(f"Get args: {descriptions}\n---\n")
    todos.extend(descriptions)
    completions.extend([False] * len(descriptions))
    return get_report()


@function_tool
async def mark_completed(index: int, completion_notes: str):
    """Mark competed the todo at given position (starting from 1) and return the full list.
    Args:
        index: The 1-based index of the todo to mark as complete,
        completion_notes: Notes about how you completed the todo in rich console markup.
    """
    print(f"Get args: index={index} {type(index)} {len(completions)}\n---\n")
    if 1 <= index <= len(completions):
        completions[index-1] = True
        show(completion_notes)
    else:
        print(f"No completion by index: {index-1}")
    return get_report()

In [8]:
agent = Agent(name='Problem Solver', 
              instructions=system_prompt, 
              model='gpt-5-mini',
              tools=[create_todos, mark_completed]
             )

In [9]:
todos = []
completions = []
result = await Runner.run(agent, user_message)

Get args: ['Estimate distance between Boston and New York (choose a reasonable value).', 'Set up equations for positions and relative motion of the trains.', 'Solve the equation to find meeting time after 3:00 pm; convert to clock time.', 'Compute meeting location relative to Boston and present final answer.']
---



Todo #1: Estimate distance between Boston and New York (choose a reasonable value).
Todo #2: Set up equations for positions and relative motion of the trains.
Todo #3: Solve the equation to find meeting time after 3:00 pm; convert to clock time.
Todo #4: Compute meeting location relative to Boston and present final answer.

Get args: index=1 <class 'int'> 4
---



Chose a reasonable estimate for the distance: Boston–New York ≈ 215 miles (typical driving distance; reasonable for
this puzzle).

Todo #1: Estimate distance between Boston and New York (choose a reasonable value).
Todo #2: Set up equations for positions and relative motion of the trains.
Todo #3: Solve the equation to find meeting time after 3:00 pm; convert to clock time.
Todo #4: Compute meeting location relative to Boston and present final answer.

...

Solve 60*(t+1)+80*t = 215 → 60t+60+80t = 215 → 140t + 60 = 215 → 140t = 155 → t = 155/140 = 31/28 ≈ 1.107142857 
hours after 3:00 pm. Convert: 0.107142857 hr = 0.107142857*60 = 6.4285714 min = 6 min + 0.4285714*60 sec ≈ 6 min 
25.7 sec. Meeting time ≈ 3:00 pm + 1 hr 6 min 25.7 sec = 4:06:25.7 pm (≈ 4:06:26 pm).

Todo #1: Estimate distance between Boston and New York (choose a reasonable value).
Todo #2: Set up equations for positions and relative motion of the trains.
Todo #3: Solve the equation to find meeting time after 3:00 pm; convert to clock time.
Todo #4: Compute meeting location relative to Boston and present final answer.

Get args: index=4 <class 'int'> 4
---



Distance traveled by Train A from Boston to meeting point: 60*(t+1) = 60*(31/28 + 1?) Wait compute t+1 = 1 + 31/28 
= (28/28 +31/28)=59/28 ≈2.107142857. Multiply by 60 → 126.4285714 miles from Boston. So meeting point ≈126.43 miles
from Boston (215 - 126.4285714 = 88.5714286 miles from New York). Final answer: They meet at about 4:06:26 pm, 
about 126.43 miles from Boston.

Todo #1: Estimate distance between Boston and New York (choose a reasonable value).
Todo #2: Set up equations for positions and relative motion of the trains.
Todo #3: Solve the equation to find meeting time after 3:00 pm; convert to clock time.
Todo #4: Compute meeting location relative to Boston and present final answer.

In [10]:
show(result.final_output)

Solution (rich console output)

Assumption:
- Distance Boston ↔ New York ≈ 215 miles (reasonable estimate).

Setup:
- Train A (Boston) leaves 2:00 pm at 60 mph.
- Train B (New York) leaves 3:00 pm at 80 mph toward Boston.
- Let t = hours after 3:00 pm when they meet.
- Train A distance by meeting = 60*(t + 1) miles (left 1 hour earlier).
- Train B distance by meeting = 80*t miles.
- Condition: 60*(t + 1) + 80*t = 215.

Calculation:
- 60t + 60 + 80t = 215 → 140t + 60 = 215 → 140t = 155 → t = 155/140 = 31/28 ≈ 1.107142857 hours after 3:00 pm.
- 0.107142857 hr = 0.107142857*60 = 6.4285714 min = 6 min 25.7 sec.

Answer:
- They meet at approximately 4:06:26 pm.
- Meeting location: ≈ 60*(t+1) = 60*(59/28) = 126.4286 miles from Boston (≈ 88.5714 miles from New York).

[non-fatal] Tracing: server error 504, retrying.
